In [1]:
import pandas as pd

# Echten Datensatz direkt aus dem Internet laden (Titanic-Datensatz)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.head())  # Zeigt die ersten 5 Zeilen des DataFrames an

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [2]:
# Erster Überblick — immer der erste Schritt!
print("Form (Zeilen, Spalten):", df.shape)
print()
print(df.head())
print()
df.info()

Form (Zeilen, Spalten): (891, 12)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            

## Fehlen de Werte zählen

In [ ]:
# Absolute Anzahl je Spalte
print("Fehlende Werte je Spalte:")
print(df.isnull().sum())
print()

Fehlende Werte je Spalte:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64



In [4]:
# Naiver Ansatz — zeigt, wie viel man verlieren würde
print("Vorher:", df.shape)
print("Nach dropna():", df.dropna().shape)

Vorher: (891, 12)
Nach dropna(): (183, 12)


In [5]:
# ===== Strategie 1: Spalte mit zu vielen Fehlwerten entfernen =====

# Cabin: ~77% fehlen → Spalte löschen (axis=1 bedeutet Spalte)
df = df.drop("Cabin", axis=1)


# ===== Strategie 2: Numerische Spalte imputieren =====

# Age: Median statt Mittelwert, da Ausreißer den Mittelwert verzerren
median_age = df["Age"].median()
df["Age"] = df["Age"].fillna(median_age)
print("Verwendeter Median:", median_age)


# ===== Strategie 3: Kategorische Spalte imputieren =====

# Embarked: Modus (häufigster Wert) — bei nur 2 Fehlwerten unproblematisch
mode_embarked = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(mode_embarked)
print("Häufigster Hafen:", mode_embarked)


# ===== Kontrolle =====
print("\nFehlende Werte nach der Bereinigung:")
print(df.isnull().sum())
print("Form:", df.shape)

Verwendeter Median: 28.0
Häufigster Hafen: S

Fehlende Werte nach der Bereinigung:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64
Form: (891, 11)


In [6]:
print("Mittelwert:", df["Age"].mean())
print("Median:", df["Age"].median())

Mittelwert: 29.36158249158249
Median: 28.0


In [7]:
# ===== Duplikate identifizieren =====

# Wie viele komplett doppelte Zeilen gibt es?
print("Anzahl doppelter Zeilen:", df.duplicated().sum())

# Duplikate nur anhand bestimmter Spalten prüfen
print("Doppelte Namen:", df.duplicated(subset=["Name"]).sum())

Anzahl doppelter Zeilen: 0
Doppelte Namen: 0


In [8]:
# ===== Übung mit erzeugten Duplikaten =====

data = {
    "Name": ["Alice", "Bob", "Alice", "David", "Bob"],
    "Age": [25, 30, 25, 22, 30]
}
df_dup = pd.DataFrame(data)

# Boolesche Series: Welche Zeile ist eine Wiederholung?
print(df_dup.duplicated())

# Standard: erstes Vorkommen behalten
print(df_dup.drop_duplicates())

# Letztes Vorkommen behalten
print(df_dup.drop_duplicates(keep="last"))

# Alle Vorkommen entfernen — auch das Original!
print(df_dup.drop_duplicates(keep=False))

0    False
1    False
2     True
3    False
4     True
dtype: bool
    Name  Age
0  Alice   25
1    Bob   30
3  David   22
    Name  Age
2  Alice   25
3  David   22
4    Bob   30
    Name  Age
3  David   22


In [9]:
# ===== Zeitreihe mit Lücken =====

data = {"Value": [1, None, 3, None, 5]}
df_ts = pd.DataFrame(data)

# Lineare Interpolation — schätzt den Wert zwischen den Nachbarn
print(df_ts["Value"].interpolate(method="linear"))

# Forward Fill — der letzte bekannte Wert wird nach vorne getragen
print(df_ts["Value"].ffill())

# Backward Fill — der nächste bekannte Wert wird nach hinten getragen
print(df_ts["Value"].bfill())

0    1.0
1    2.0
2    3.0
3    4.0
4    5.0
Name: Value, dtype: float64
0    1.0
1    1.0
2    3.0
3    3.0
4    5.0
Name: Value, dtype: float64
0    1.0
1    3.0
2    3.0
3    5.0
4    5.0
Name: Value, dtype: float64
